## Setup
<a id="setup"></a>

In [1]:
%load_ext autoreload
%autoreload 2

In [23]:
from pathlib import Path

path = Path(".") / "data" / "cityscapes"
path_dir_split = path / "leftImg8bit_sequence" / "train"
paths_dir_city = [path_dir_city for path_dir_city in sorted(path_dir_split.iterdir()) if path_dir_city.is_dir()]
names_cities = [path_dir_city.name for path_dir_city in paths_dir_city]
names_cities

['aachen',
 'bochum',
 'bremen',
 'cologne',
 'darmstadt',
 'dusseldorf',
 'erfurt',
 'hamburg',
 'hanover',
 'jena',
 'krefeld',
 'monchengladbach',
 'strasbourg',
 'stuttgart',
 'tubingen',
 'ulm',
 'weimar',
 'zurich']

### Config
<a id="setup_config"></a>

In [4]:
import self_supervised_learning_of_depth_and_motion.config as config

config.list_available()

['cityscapes_unnormalized',
 'lfw_trinet',
 'lfw_trinet_pretrained',
 'lfw_trinet_pretrained_shnmining',
 'lfw_unnormalized']

### Modules
<a id="setup_modules"></a>

In [ ]:
from pathlib import Path

import numpy as np
import torch
import torchinfo
import torchvision as tv

import self_supervised_learning_of_depth_and_motion.scripts.init_exp as init_exp
import self_supervised_learning_of_depth_and_motion.scripts.compute_mean_and_std as compute_mean_and_std
from self_supervised_learning_of_depth_and_motion.evaluation.evaluator import Evaluator
from self_supervised_learning_of_depth_and_motion.training.trainer import Trainer
import self_supervised_learning_of_depth_and_motion.libs.factory as factory
import self_supervised_learning_of_depth_and_motion.libs.utils_checkpoints as utils_checkpoints
import self_supervised_learning_of_depth_and_motion.libs.utils_data as utils_data
import self_supervised_learning_of_depth_and_motion.libs.utils_model as utils_model
import self_supervised_learning_of_depth_and_motion.visualization.plot as plot
import self_supervised_learning_of_depth_and_motion.visualization.visualize as visualize

### Paths and names
<a id="setup_paths_and_names"></a>

In [ ]:
name_exp_cityscapes_unnormalized = "cityscapes_unnormalized"

path_dir_exp_cityscapes_unnormalized = Path(config._PATH_DIR_EXPS) / name_exp_cityscapes_unnormalized

## Initialization

In [ ]:
init_exp.init_exp(name_exp=name_exp_lfw_trinet, name_config=name_exp_lfw_trinet, use_remove_old=False)

In [ ]:
init_exp.init_exp(name_exp=name_exp_lfw_trinet_pretrained, name_config=name_exp_lfw_trinet_pretrained, use_remove_old=False)

In [ ]:
init_exp.init_exp(name_exp=name_exp_lfw_trinet_pretrained_shnmining, name_config=name_exp_lfw_trinet_pretrained_shnmining, use_remove_old=False)

## Data

### LFW

In [ ]:
compute_mean_and_std.compute_mean_and_std(name_config=name_exp_lfw_unnormalized, split="training")

In [ ]:
config.set_config_exp(path_dir_exp_lfw_trinet)

splits = ["test", "validation", "training", "training"]
uses_unnormalize = [True, True, True, False]

for split, use_unnormalize in zip(splits, uses_unnormalize):
    features, target = utils_data.sample(split=split, num_samples=16, use_unnormalize=use_unnormalize, use_labelset=True)
    values = list(features.values())
    features = torch.stack(tuple(tv.utils.make_grid([value[i] for value in values]) for i in range(len(values[0]))), dim=0)
    target = target["anchor"]

    path_save = path_dir_exp_lfw_trinet / "visualizations" / f"Sample_{split}{"" if use_unnormalize else "_normalized"}.png"
    visualize.visualize_images(features, labels=target, path_save=path_save)

## Models

### Adapted TriNet

In [ ]:
config.set_config_exp(path_dir_exp_lfw_trinet)

model = factory.create_model()
print(model)

input_dummy = dict(anchor=torch.zeros(config.MODEL["shape_input"]), positive=torch.zeros(config.MODEL["shape_input"]), negative=torch.zeros(config.MODEL["shape_input"]))
print(torchinfo.summary(model, input_data=[input_dummy], verbose=0, col_names=("input_size", "output_size", "params_percent"), mode="eval"))

## Experiments

### Adapted TriNet on LFW

In [ ]:
config.set_config_exp(path_dir_exp_lfw_trinet)

trainer = Trainer(name_exp_lfw_trinet)
trainer.loop(config.TRAINING["num_epochs"])

log = trainer.log
path_plots = path_dir_exp_lfw_trinet / "plots"
plot.plot_loss(log, path_save=path_plots / "Loss.png")
plot.plot_metrics(log, path_plots=path_plots)
plot.plot_learning_rate(log, path_save=path_plots / "Learning_rate.png")

In [ ]:
config.set_config_exp(path_dir_exp_lfw_trinet)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset, dataloader = factory.create_dataset_and_dataloader(split="test")

num_clusters = 10
counts_labels = torch.bincount(dataset.targets, minlength=len(dataset.labelset))
topk = torch.topk(counts_labels, num_clusters)

targets_topk = topk.indices
indices_topk = dataset.indices[torch.isin(dataset.targets, targets_topk)]
list_features, list_targets = utils_data.sample_dataset(dataset, indices_topk)

model = utils_checkpoints.load_model(path_dir_exp_lfw_trinet / "checkpoints" / "best.pth")
model = model.to(device)

featuress_anchor = []
latents_anchor = []
targets_anchor = []
with torch.no_grad():
    for features, targets in zip(list_features, list_targets):
        features_anchor = features["anchor"][None, ...]
        target_anchor = torch.as_tensor(targets["anchor"])[None, ...]

        featuress_anchor += [features_anchor]
        targets_anchor += [target_anchor]

        features_anchor = features_anchor.to(device)
        latent_anchor = model.forward_single(features_anchor)
        latents_anchor += [latent_anchor.cpu()]

featuress_anchor_flat = np.concatenate([features_anchor.flatten(1) for features_anchor in featuress_anchor])
featuress_anchor = np.concatenate([utils_data.unnormalize(features_anchor) for features_anchor in featuress_anchor])
targets_anchor = np.concatenate(targets_anchor)
latents_anchor = np.concatenate(latents_anchor)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

labels_kmeans_features = KMeans(n_clusters=num_clusters).fit_predict(featuress_anchor_flat)
labels_kmeans_latents = KMeans(n_clusters=num_clusters).fit_predict(latents_anchor)

ari_imgs = adjusted_rand_score(targets_anchor, labels_kmeans_features)
ari_embs = adjusted_rand_score(targets_anchor, labels_kmeans_latents)

print(f"Clustering images achieves  ARI={round(ari_imgs*100,2)}%")
print(f"Clustering embeddings achieves ARI={round(ari_embs*100,2)}%")

print(f"Compression ratio: {latents_anchor.shape[-1]}/{featuress_anchor_flat.shape[-1]}  = {round(latents_anchor.shape[-1]/featuress_anchor_flat.shape[-1] * 100, 2)}%")

In [ ]:
path_save = path_dir_exp_lfw_trinet / "plots" / "Projection_pca_samples.png"
plot.plot_projection_pca(featuress_anchor_flat, num_points=1000, targets=targets_anchor, labelset=dataset.labelset, images_annotation=featuress_anchor, zoom_annotation=0.1, path_save=path_save)

path_save = path_dir_exp_lfw_trinet / "plots" / "Projection_pca_latent.png"
plot.plot_projection_pca(latents_anchor, num_points=1000, targets=targets_anchor, labelset=dataset.labelset, images_annotation=featuress_anchor, zoom_annotation=0.1, path_save=path_save)

path_save = path_dir_exp_lfw_trinet / "plots" / "Projection_tsne_samples.png"
plot.plot_projection_tsne(featuress_anchor_flat, num_points=1000, targets=targets_anchor, labelset=dataset.labelset, images_annotation=featuress_anchor, zoom_annotation=0.1, path_save=path_save)

path_save = path_dir_exp_lfw_trinet / "plots" / "Projection_tsne_latent.png"
plot.plot_projection_tsne(latents_anchor, num_points=1000, targets=targets_anchor, labelset=dataset.labelset, images_annotation=featuress_anchor, zoom_annotation=0.1, path_save=path_save)

### Pretrained TriNet on LFW

In [ ]:
config.set_config_exp(path_dir_exp_lfw_trinet_pretrained)

trainer = Trainer(name_exp_lfw_trinet_pretrained)
trainer.loop(config.TRAINING["num_epochs"])

log = trainer.log
path_plots = path_dir_exp_lfw_trinet_pretrained / "plots"
plot.plot_loss(log, path_save=path_plots / "Loss.png")
plot.plot_metrics(log, path_plots=path_plots)
plot.plot_learning_rate(log, path_save=path_plots / "Learning_rate.png")

In [ ]:
config.set_config_exp(path_dir_exp_lfw_trinet_pretrained)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset, dataloader = factory.create_dataset_and_dataloader(split="test")

num_clusters = 10
counts_labels = torch.bincount(dataset.targets, minlength=len(dataset.labelset))
topk = torch.topk(counts_labels, num_clusters)

targets_topk = topk.indices
indices_topk = dataset.indices[torch.isin(dataset.targets, targets_topk)]
list_features, list_targets = utils_data.sample_dataset(dataset, indices_topk)

model = utils_checkpoints.load_model(path_dir_exp_lfw_trinet_pretrained / "checkpoints" / "best.pth")
model = model.to(device)

featuress_anchor = []
latents_anchor = []
targets_anchor = []
with torch.no_grad():
    for features, targets in zip(list_features, list_targets):
        features_anchor = features["anchor"][None, ...]
        target_anchor = torch.as_tensor(targets["anchor"])[None, ...]

        featuress_anchor += [features_anchor]
        targets_anchor += [target_anchor]

        features_anchor = features_anchor.to(device)
        latent_anchor = model.forward_single(features_anchor)
        latents_anchor += [latent_anchor.cpu()]

featuress_anchor_flat = np.concatenate([features_anchor.flatten(1) for features_anchor in featuress_anchor])
featuress_anchor = np.concatenate([utils_data.unnormalize(features_anchor) for features_anchor in featuress_anchor])
targets_anchor = np.concatenate(targets_anchor)
latents_anchor = np.concatenate(latents_anchor)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

labels_kmeans_features = KMeans(n_clusters=num_clusters).fit_predict(featuress_anchor_flat)
labels_kmeans_latents = KMeans(n_clusters=num_clusters).fit_predict(latents_anchor)

ari_imgs = adjusted_rand_score(targets_anchor, labels_kmeans_features)
ari_embs = adjusted_rand_score(targets_anchor, labels_kmeans_latents)

print(f"Clustering images achieves  ARI={round(ari_imgs*100,2)}%")
print(f"Clustering embeddings achieves ARI={round(ari_embs*100,2)}%")

print(f"Compression ratio: {latents_anchor.shape[-1]}/{featuress_anchor_flat.shape[-1]}  = {round(latents_anchor.shape[-1]/featuress_anchor_flat.shape[-1] * 100, 2)}%")

In [ ]:
path_save = path_dir_exp_lfw_trinet_pretrained / "plots" / "Projection_pca_samples.png"
plot.plot_projection_pca(featuress_anchor_flat, num_points=1000, targets=targets_anchor, labelset=dataset.labelset, images_annotation=featuress_anchor, zoom_annotation=0.1, path_save=path_save)

path_save = path_dir_exp_lfw_trinet_pretrained / "plots" / "Projection_pca_latent.png"
plot.plot_projection_pca(latents_anchor, num_points=1000, targets=targets_anchor, labelset=dataset.labelset, images_annotation=featuress_anchor, zoom_annotation=0.1, path_save=path_save)

path_save = path_dir_exp_lfw_trinet_pretrained / "plots" / "Projection_tsne_samples.png"
plot.plot_projection_tsne(featuress_anchor_flat, num_points=1000, targets=targets_anchor, labelset=dataset.labelset, images_annotation=featuress_anchor, zoom_annotation=0.1, path_save=path_save)

path_save = path_dir_exp_lfw_trinet_pretrained / "plots" / "Projection_tsne_latent.png"
plot.plot_projection_tsne(latents_anchor, num_points=1000, targets=targets_anchor, labelset=dataset.labelset, images_annotation=featuress_anchor, zoom_annotation=0.1, path_save=path_save)

### Pretrained TriNet on LFW with semi-hard negative mining

In [ ]:
config.set_config_exp(path_dir_exp_lfw_trinet_pretrained_shnmining)

trainer = Trainer(name_exp_lfw_trinet_pretrained_shnmining)
trainer.loop(config.TRAINING["num_epochs"])

log = trainer.log
path_plots = path_dir_exp_lfw_trinet_pretrained_shnmining / "plots"
plot.plot_loss(log, path_save=path_plots / "Loss.png")
plot.plot_metrics(log, path_plots=path_plots)
plot.plot_learning_rate(log, path_save=path_plots / "Learning_rate.png")

In [ ]:
config.set_config_exp(path_dir_exp_lfw_trinet_pretrained_shnmining)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset, dataloader = factory.create_dataset_and_dataloader(split="test")

counts_labels = torch.bincount(dataset.targets, minlength=len(dataset.labelset))
topk = torch.topk(counts_labels, 10)

targets_topk = topk.indices
indices_topk = dataset.indices[torch.isin(dataset.targets, targets_topk)]
list_features, list_targets = utils_data.sample_dataset(dataset, indices_topk)

model = utils_checkpoints.load_model(path_dir_exp_lfw_trinet_pretrained_shnmining / "checkpoints" / "best.pth")
model = model.to(device)

featuress_anchor = []
latents_anchor = []
targets_anchor = []
with torch.no_grad():
    for features, targets in zip(list_features, list_targets):
        features_anchor = features["anchor"][None, ...]
        target_anchor = torch.as_tensor(targets["anchor"])[None, ...]

        featuress_anchor += [features_anchor]
        targets_anchor += [target_anchor]

        features_anchor = features_anchor.to(device)
        latent_anchor = model.forward_single(features_anchor)
        latents_anchor += [latent_anchor.cpu()]

featuress_anchor_flat = np.concatenate([features_anchor.flatten(1) for features_anchor in featuress_anchor])
featuress_anchor = np.concatenate([utils_data.unnormalize(features_anchor) for features_anchor in featuress_anchor])
targets_anchor = np.concatenate(targets_anchor)
latents_anchor = np.concatenate(latents_anchor)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

labels_kmeans_features = KMeans(n_clusters=10).fit_predict(featuress_anchor_flat)
labels_kmeans_latents = KMeans(n_clusters=10).fit_predict(latents_anchor)

ari_imgs = adjusted_rand_score(targets_anchor, labels_kmeans_features)
ari_embs = adjusted_rand_score(targets_anchor, labels_kmeans_latents)

print(f"Clustering images achieves  ARI={round(ari_imgs*100,2)}%")
print(f"Clustering embeddings achieves ARI={round(ari_embs*100,2)}%")

print(f"Compression ratio: {latents_anchor.shape[-1]}/{featuress_anchor_flat.shape[-1]}  = {round(latents_anchor.shape[-1]/featuress_anchor_flat.shape[-1] * 100, 2)}%")

In [ ]:
path_save = path_dir_exp_lfw_trinet_pretrained_shnmining / "plots" / "Projection_pca_samples.png"
plot.plot_projection_pca(featuress_anchor_flat, num_points=1000, targets=targets_anchor, labelset=dataset.labelset, images_annotation=featuress_anchor, zoom_annotation=0.1, path_save=path_save)

path_save = path_dir_exp_lfw_trinet_pretrained_shnmining / "plots" / "Projection_pca_latent.png"
plot.plot_projection_pca(latents_anchor, num_points=1000, targets=targets_anchor, labelset=dataset.labelset, images_annotation=featuress_anchor, zoom_annotation=0.1, path_save=path_save)

path_save = path_dir_exp_lfw_trinet_pretrained_shnmining / "plots" / "Projection_tsne_samples.png"
plot.plot_projection_tsne(featuress_anchor_flat, num_points=1000, targets=targets_anchor, labelset=dataset.labelset, images_annotation=featuress_anchor, zoom_annotation=0.1, path_save=path_save)

path_save = path_dir_exp_lfw_trinet_pretrained_shnmining / "plots" / "Projection_tsne_latent.png"
plot.plot_projection_tsne(latents_anchor, num_points=1000, targets=targets_anchor, labelset=dataset.labelset, images_annotation=featuress_anchor, zoom_annotation=0.1, path_save=path_save)